In [0]:
from pyspark.sql.functions import col, current_timestamp, input_file_name
raw_path = "/Volumes/crimeworkspace/default/raw"
bronze_table = "crimeworkspace.bronze.bronze_crime"
checkpoint_path = "/Volumes/crimeworkspace/default/checkpoints/bronze_crime"


In [0]:
df_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", True)
    .option("cloudFiles.schemaLocation", checkpoint_path + "/schema")
    .option("cloudFiles.inferColumnTypes", "True")
    .load(raw_path)
    )

In [0]:
df_bronze = (
    df_raw
    .withColumn("ingest_ts", current_timestamp())
    .withColumn("source_file",  col("_metadata.file_path"))
)


In [0]:
df_bronze = df_bronze.toDF(*[
    c.replace(' ', '').replace('-', '') 
    for c in df_bronze.columns
])

In [0]:
query = (
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)  # Process all new files then stop
    .toTable(bronze_table)
)
query.awaitTermination()

In [0]:
df_bronze_table = spark.read.table(bronze_table)
display(df_bronze_table)

DR_NO,DateRptd,DATEOCC,TIMEOCC,AREA,AREANAME,RptDistNo,Part12,CrmCd,CrmCdDesc,Mocodes,VictAge,VictSex,VictDescent,PremisCd,PremisDesc,WeaponUsedCd,WeaponDesc,Status,StatusDesc,CrmCd1,CrmCd2,CrmCd3,CrmCd4,LOCATION,CrossStreet,LAT,LON,_rescued_data,ingest_ts,source_file
200904022,2020-01-01,01/01/2020 12:00:00 AM,1045,9,Van Nuys,966,2,626,INTIMATE PARTNER - SIMPLE ASSAULT,2000 0400 0416 0444 1309,39,F,W,502.0,"MULTI-UNIT DWELLING (APARTMENT, DUPLEX, ETC)",400.0,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",AO,Adult Other,626.0,null,null,null,13300 HUSTON ST,null,34.1595,-118.4225,null,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv
201304045,2020-01-01,01/01/2020 12:00:00 AM,2000,13,Newton,1309,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",1822 0411 0449,44,M,H,102.0,SIDEWALK,500.0,UNKNOWN WEAPON/OTHER WEAPON,IC,Invest Cont,230.0,null,null,null,7TH ST,DECATUR ST,34.0348,-118.235,null,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv
200204046,2020-01-01,01/01/2020 12:00:00 AM,335,2,Rampart,216,2,740,"VANDALISM - FELONY ($400 & OVER, ALL CHURCH VANDALISMS)",1822 0321 0329,0,X,X,401.0,MINI-MART,null,null,IC,Invest Cont,740.0,null,null,null,2200 W SUNSET BL,null,34.0778,-118.2658,null,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv
201204054,2020-01-01,01/01/2020 12:00:00 AM,200,12,77th Street,1269,1,210,ROBBERY,1206 0371 0416 0446 0417 0344 1309 0342 0906,31,M,H,501.0,SINGLE FAMILY DWELLING,400.0,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",IC,Invest Cont,210.0,null,null,null,1000 E 84TH PL,null,33.962,-118.2575,null,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv
201204077,2020-01-01,01/01/2020 12:00:00 AM,2025,12,77th Street,1203,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",1309 0913 0411,31,F,B,102.0,SIDEWALK,207.0,OTHER KNIFE,IC,Invest Cont,230.0,null,null,null,1600 W 47TH ST,null,34.001,-118.3067,null,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv
201504058,2020-01-01,01/01/2020 12:00:00 AM,2230,15,N Hollywood,1591,1,350,"THEFT, PERSON",0344 0400,17,M,W,228.0,BOWLING ALLEY*,null,null,IC,Invest Cont,350.0,null,null,null,12600 VENTURA BL,null,34.1437,-118.4074,null,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv
200204029,2020-01-01,01/01/2020 12:00:00 AM,1355,2,Rampart,212,2,740,"VANDALISM - FELONY ($400 & OVER, ALL CHURCH VANDALISMS)",0329 0601 2004 2001,0,X,X,517.0,MISSIONS/SHELTERS,null,null,AO,Adult Other,740.0,null,null,null,300 N MADISON AV,null,34.0764,-118.2892,null,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv
202104033,2020-01-01,01/01/2020 12:00:00 AM,2140,21,Topanga,2136,2,901,VIOLATION OF RESTRAINING ORDER,2038,14,M,H,501.0,SINGLE FAMILY DWELLING,null,null,IC,Invest Cont,901.0,null,null,null,7500 JORDAN AV,null,34.2065,-118.6029,null,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv
200704041,2020-01-01,01/01/2020 12:00:00 AM,1210,7,Wilshire,724,1,236,INTIMATE PARTNER - AGGRAVATED ASSAULT,2000 0416 1218 0913 2004,41,F,B,510.0,NURSING/CONVALESCENT/RETIREMENT HOME,304.0,CLUB/BAT,IC,Invest Cont,236.0,null,null,null,100 S GARDNER ST,null,34.0735,-118.3571,null,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv
201104052,2020-01-01,01/01/2020 12:00:00 AM,2030,11,Northeast,1105,2,740,"VANDALISM - FELONY ($400 & OVER, ALL CHURCH VANDALISMS)",2004 1402 1020 1307 0329,0,X,X,502.0,"MULTI-UNIT DWELLING (APARTMENT, DUPLEX, ETC)",null,null,AA,Adult Arrest,740.0,998.0,null,null,2400 COLORADO BL,null,34.1442,-118.218,null,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv


In [0]:
df_bronze_table.count()

192708